In [1]:
import numpy as np
import pandas as pd
import torch.nn.functional as F
from torch.nn import *
import torch
from torch.utils.data import IterableDataset
from tqdm import tqdm
from torch_geometric.data import HeteroData
from torch_geometric.loader import DataLoader
from constants import *
from model_gnn import GNN
import gc
import itertools

# DEVICE1/DEVICE2 imported from constants

In [2]:
el_df = pd.read_parquet('data/el_db.parquet')
el_df = el_df[el_df['el'] == 1].drop(columns=['el'])
el_df = el_df[~el_df['allele'].str.contains(',')]
el_df = el_df.loc[el_df['allele'].str.contains('HLA-[ABC]', regex=True)]
el_df = el_df.groupby('allele').filter(lambda x: len(x) >= 100)
el_df['peptide'] = ['[CLS] ' + ' '.join(list(peptide)) + ' [SEP]' for peptide in el_df['peptide'].values]
unique_alleles = pd.Series(el_df['allele'].unique())
el_df

,peptide,source,allele
1,[CLS] A T S R T Y L Y R [SEP],netmhcpan_el,HLA-A*11:01
8,[CLS] V S K G T L V Q T K [SEP],netmhcpan_el,HLA-A*11:01
9,[CLS] Y L T K K F A E L [SEP],netmhcpan_el,HLA-A*02:07
11,[CLS] Y L F D L P L K V [SEP],netmhcpan_el,HLA-A*02:01
13,[CLS] S A V P K V M K V [SEP],netmhcpan_el,HLA-C*12:03
...,...,...,...
13369322,[CLS] F S I P C E P N L [SEP],mhcmotifatlas,HLA-C*17:01
13369323,[CLS] F S L M V V N R L [SEP],mhcmotifatlas,HLA-C*17:01
13369324,[CLS] F S C G V I S T L [SEP],mhcmotifatlas,HLA-C*17:01
13369325,[CLS] F S V C H I L L L [SEP],mhcmotifatlas,HLA-C*17:01


In [3]:
mhc_df = pd.read_parquet('data/mhc_db.parquet')
mhc_df = mhc_df.loc[mhc_df['allele'].isin(unique_alleles)]
el_df = el_df.loc[el_df['allele'].isin(mhc_df['allele'].values)]
unique_alleles = pd.Series(el_df['allele'].unique())
mhc_df['sequence'] = mhc_df['sequence'].apply(lambda x: ' '.join(list(x)))
mhc_homo = pd.DataFrame([['HLA-A*homozygous', 'HLA-A*homozygous'], ['HLA-B*homozygous', 'HLA-B*homozygous'], ['HLA-C*homozygous', 'HLA-C*homozygous']], columns=mhc_df.columns)
mhc_df = pd.concat([mhc_df, mhc_homo])
mhc_df['sequence'] = '[CLS] ' + mhc_df['sequence'] + ' [SEP]'
mhc_df

,allele,sequence
1077,HLA-A*01:01,[CLS] M A V M A P R T L L L L L S G A L A L T ...
1349,HLA-A*02:01,[CLS] M A V M A P R T L V L L L S G A L A L T ...
1350,HLA-A*02:02,[CLS] M A V M A P R T L V L L L S G A L A L T ...
1351,HLA-A*02:03,[CLS] M A V M A P R T L V L L L S G A L A L T ...
1352,HLA-A*02:04,[CLS] M A V M A P R T L V L L L S G A L A L T ...
...,...,...
10775,HLA-C*16:02,[CLS] M R V M A P R T L I L L L S G A L A L T ...
10915,HLA-C*17:01,[CLS] M R V M A P Q A L L L L L S G A L A L I ...
0,HLA-A*homozygous,[CLS] HLA-A*homozygous [SEP]
1,HLA-B*homozygous,[CLS] HLA-B*homozygous [SEP]


In [7]:
MAX_LEN_MHC = mhc_df['sequence'].str.split().str.len().max()
MAX_LEN_PEP = pd.Series(el_df['peptide'].unique()).str.split().str.len().max()
MAX_LEN_PEP, MAX_LEN_MHC

(np.int64(25), np.int64(368))

In [8]:
mhc_features = np.array([
    [TOKEN_VOCABULARY[e] for e in seq.split()] + [0 for _ in range(MAX_LEN_MHC - len(seq.split()))] for seq in mhc_df['sequence'].values
])
mhc_x = torch.tensor(mhc_features, dtype=torch.int32)
mhc_mhc_edges = torch.concat([torch.arange(len(mhc_features)).unsqueeze(0), torch.arange(len(mhc_features)).unsqueeze(0)], 0)  # MHCs only connected to themselves

In [9]:
def random_sample_generator(alleles, num_samples=100, peptides_per_allele=200):
    loci = pd.Series(alleles).str[4].unique()
    locus_dict = {l: pd.Series(alleles[alleles.str.contains(f'HLA-{l}')]) for l in loci}
    for _ in range(num_samples):
        sampled_alleles = np.concatenate([a.sample(2, replace=True).values for a in locus_dict.values()])
        df = el_df.loc[el_df['allele'].isin(sampled_alleles)]
        df = df.groupby('allele').apply(lambda x: x.sample(min([peptides_per_allele, len(x)])), include_groups=False).reset_index()
        df = df[['allele', 'peptide']].drop_duplicates(subset='peptide')
        n_alleles_per_locus = pd.Series(np.unique(sampled_alleles)).str[4].value_counts()
        df['sample_alleles'] = ','.join(sorted(np.unique(np.concatenate([[f'HLA-{l}*homozygous' for l in loci if n_alleles_per_locus[l] == 1], sampled_alleles]))))
        yield df


def get_hetero_data(fn, *args, **kwargs):
    for df in fn(*args, **kwargs):
        sample_peptide_features = np.array([
            [TOKEN_VOCABULARY[e] for e in seq.split()] + [0 for _ in range(MAX_LEN_PEP - len(seq.split()))] for seq in df['peptide'].values
        ])
        entry = HeteroData()
        entry['peptide'].x = torch.tensor(sample_peptide_features, dtype=torch.int32)
        entry['mhc'].x = mhc_x
        entry['mhc'].y = torch.tensor(mhc_df['allele'].isin(df['sample_alleles'].iloc[0].split(',')).values * 1, dtype=torch.float32).unsqueeze(1)
        peptide_mhc_edges = torch.tensor(np.array(list(itertools.product(range(len(entry['peptide'].x)), range(len(mhc_features))))).T, dtype=torch.int32)
        peptide_peptide_edges = torch.concat([torch.arange(len(entry['peptide'].x)).unsqueeze(0), torch.arange(len(entry['peptide'].x)).unsqueeze(0)], 0)
        entry['peptide', 'determines', 'mhc'].edge_index = peptide_mhc_edges
        entry['peptide', 'influences', 'peptide'].edge_index = peptide_peptide_edges
        entry['mhc', 'influences', 'mhc'].edge_index = mhc_mhc_edges
        yield entry


class HeteroGeneratorDataset(IterableDataset):
    def __init__(self, generator_fn, *args, **kwargs):
        super().__init__()
        self.generator_fn = get_hetero_data(generator_fn, *args, **kwargs)

    def __iter__(self):
        return iter(self.generator_fn)

## Training

In [ ]:
gc.collect()
torch.cuda.empty_cache()

def typing_acc(batch, pred):
    tmp = pd.DataFrame(np.concatenate([
        pred.cpu().detach().numpy(),
        batch['mhc'].y.cpu().detach().numpy(),
        batch['mhc'].batch.unsqueeze(1)
    ], axis=1), columns=['y_pred', 'y', 'batch'])
    tmp['allele'] = np.concatenate(np.repeat([mhc_df['allele'].values], batch.batch_size, axis=0))
    tmp['locus'] = tmp['allele'].str[4]
    
    tmp = tmp.groupby(['batch', 'locus']).apply(
        lambda x: pd.Series({
            'y': ','.join(x.sort_values('y')['allele'][-2:].values),
            'y_pred': ','.join(x.sort_values('y_pred')['allele'][-2:].values),
        }), include_groups=False
    ).reset_index()
    
    return tmp.groupby('batch').apply(
        lambda x: len(set(','.join(x['y']).split(',')).intersection(set(','.join(x['y_pred']).split(',')))) / 6,
        include_groups=False
    ).mean()


NAME = 'pretrain_mhc_ba_el_in-silico'
BATCH_SIZE = 4
NUM_SAMPLES = 4_000
PEPTIDES_PER_ALLELE = 200
LAST_N_EPOCHS = 4  # epochs for early stopping

model = GNN(
    vocab_size=45,
    embedding_dim=128,
    dim_ff_enc_pep=128,
    n_heads_enc_pep=8,
    n_layers_enc_pep=6,
    dim_ff_enc_mhc=1024,
    n_heads_enc_mhc=8,
    n_layers_enc_mhc=6,
    n_heads_conv=8,
    n_layers_conv=2,
    dim_out_conv=32,
    dropout=0.1,
    act=F.leaky_relu,
    device1=DEVICE1,
    device2=DEVICE2,
)

state_dict = torch.load('weights/pretrain_mhc_ba_el.pt')
model.load_state_dict(state_dict, strict=True)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

print(NAME)

# Training loop
epoch = 1

performance = []
while True:
    train_dataset = HeteroGeneratorDataset(random_sample_generator, alleles=unique_alleles, num_samples=NUM_SAMPLES, peptides_per_allele=PEPTIDES_PER_ALLELE)
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE)
    batches = tqdm(train_loader, total=int(np.round(NUM_SAMPLES / BATCH_SIZE, 0)))
    bl_train, train_acc = [], []
    model.train()
    for batch in batches:
        optimizer.zero_grad()
        out = model(batch)
        loss = F.binary_cross_entropy_with_logits(out, batch['mhc'].y.to(DEVICE2))
        train_acc.append(typing_acc(batch, out))
        bl_train.append(loss.item())
        loss.backward()
        optimizer.step()
        descr = f'Train epoch {epoch} - loss: {np.mean(bl_train[-100:]):.4f}, acc: {np.mean(train_acc[-100:]):.4f}'
        batches.set_description(descr)

    performance.append([epoch, np.mean(bl_train[-100:]), np.mean(train_acc[-100:])])
    performance_df = pd.DataFrame(performance, columns=['epoch', 'train_loss', 'train_acc'])
    
    if epoch > 10 and (performance_df['train_acc'].iloc[-LAST_N_EPOCHS:].max() - performance_df['train_acc'].iloc[:-LAST_N_EPOCHS].max()) < 0.001:
        torch.save(model.state_dict(), f'weights/in_silico/{NAME}.pt')
        break
    
    epoch += 1